#### Using the model from 2.2

# 3 Relative Prices and Demand

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.size': 14})

# 5 An extension of the model

### Removing subsistence-level food consumption from discretionary income

In [ ]:
class SubsistenceConsumer(ConsumerClass): #creating a subclass
    def setup(self):
        super().setup() #brings in variables from our_consumer.py
        self.par.x1_min = 3.0  # minimum subsistence quantity of food is 3 units

    def quantities(self, s1, w):
        par = self.par
        I_disc = par.I - par.p1*par.x1_min  # first reserves subsistence food (cost of food * minimum quantity) from I
        assert I_disc > 0, "income too low to cover subsistence food" # if I_disc is negative, raise an error, because the consumer cannot afford the minimum food requirement
        x1 = par.x1_min + s1*I_disc/par.p1 #total food consumption is the minimum subsistence quantity plus any discretionary share of I allocated to food
        x2 = (1-s1)*w*I_disc/par.p2 #total consumption of good 2 is the discretionary share of I allocated to it
        x3 = (1-s1)*(1-w)*I_disc/par.p3 #total consumption of good 3 is the discretionary share of I allocated to it
        return x1, x2, x3

In [ ]:
class SubsistenceConsumer(ConsumerClass): #creating a subclass
    def setup(self):
        super().setup() #brings in variables from our_consumer.py
        self.par.x1_min = 3.0  # minimum subsistence quantity of food is 3 units

    def quantities(self, s1, w):
        par = self.par
        I_disc = par.I - par.p1*par.x1_min  # first reserves subsistence food (cost of food * minimum quantity) from I
        assert I_disc > 0, "income too low to cover subsistence food" # if I_disc is negative, raise an error, because the consumer cannot afford the minimum food requirement
        x1 = par.x1_min + s1*I_disc/par.p1 #total food consumption is the minimum subsistence quantity plus any discretionary share of I allocated to food
        x2 = (1-s1)*w*I_disc/par.p2 #total consumption of good 2 (bus trips) is the discretionary share of I allocated to it
        x3 = (1-s1)*(1-w)*I_disc/par.p3 #total consumption of good 3 (train trips) is the discretionary share of I allocated to it
        return x1, x2, x3


    def solve(self, s0=None, do_print=True, **kwargs):
        """ in solve() in our_consumer.py, the reported utility is computed using a hardcoded formula that does not account for the subsistence floor. 
        This override recomputes the utility using the quantities() method, which correctly incorporates the subsistence floor.

        Recompute u here from the same (s1,w) using quantities().
        """
        opt = super().solve(s0=s0, do_print=False, **kwargs)

        x1, x2, x3 = self.quantities(opt.s1, opt.w)
        opt.u = self.utility(x1, x2, x3)

        if do_print:
            print(f"s1={opt.s1:.4f}  w={opt.w:.4f}  u={opt.u:.4f}")

        return opt

    """
Verify the model works and that the solution satisfies the budget constraint and the minimum subsistence requirement. 
The function check_subsistence_solution solves the model, calculates the quantities of goods consumed, computes the true budget shares, and checks if the budget is balanced. 
It also prints out relevant information about the solution. 
"""
from types import SimpleNamespace

def check_subsistence_solution(model, do_print=True):
    """ solve, then report quantities, true budget shares, and a budget-balance check """
    opt = model.solve(do_print=False)
    par = model.par

    x1, x2, x3 = model.quantities(opt.s1, opt.w)
    spend = par.p1*x1 + par.p2*x2 + par.p3*x3
    true_share1 = par.p1*x1/par.I
    true_share2 = par.p2*x2/par.I
    true_share3 = par.p3*x3/par.I

    result = SimpleNamespace(
        x1=x1, x2=x2, x3=x3, u=opt.u,
        nested_s1=opt.s1, nested_w=opt.w,
        true_share1=true_share1, true_share2=true_share2, true_share3=true_share3,
        spend=spend, I=par.I,
    )

    if do_print:
        print(f"x1_min = {par.x1_min:.2f}   I = {par.I:.2f}   I_disc = {par.I - par.p1*par.x1_min:.2f}")
        print(f"quantities:      x1={x1:.3f}  x2={x2:.3f}  x3={x3:.3f}")
        print(f"true budget shares (of total I): s1={true_share1:.3f}  s2={true_share2:.3f}  s3={true_share3:.3f}  (sum={true_share1+true_share2+true_share3:.4f})")
        print(f"utility = {opt.u:.4f}")
        print(f"budget check: spend={spend:.6f} vs income={par.I:.6f}  -> {'OK' if abs(spend-par.I)<1e-6 else 'MISMATCH'}")
        print(f"x1 >= x1_min ? {x1:.3f} >= {par.x1_min:.2f} -> {'OK' if x1 >= par.x1_min - 1e-9 else 'VIOLATED'}")

    return result

model = SubsistenceConsumer()
check_subsistence_solution(model)
print()